# Module 2: Data Cleaning and Structuring

## Tasks:
- Clean text: remove commas, %, +, special symbols
- Convert metrics to numeric formats
- Standardize column names
- Handle missing/null values
- Save a clean dataset for Tableau input

In [ ]:
import pandas as pd
import numpy as np
import re
import os

### 1. Load the Raw Data

In [ ]:
# Determine input path based on standard repo structure
input_path = '../Module-1 - Scraping Setup and Execution/military_raw_data.csv'
if not os.path.exists(input_path):
    # Fallback to local if running differently
    input_path = 'military_raw_data.csv'

df = pd.read_csv(input_path)
df.head()

### 2. Standardize Column Names

In [ ]:
# Convert to lowercase, strip whitespace, and replace spaces and hyphens with underscores
df.columns = df.columns.str.lower().str.strip().str.replace(r'\s+', '_', regex=True).str.replace('-', '_')
print(df.columns.tolist())

### 3. Clean Text and Convert Metrics to Numeric Formats

In [ ]:
text_cols = ['country', 'continent', 'region', 'alliance']

def clean_currency_and_numbers(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        # Remove non-numeric characters except period
        val = re.sub(r'[^\d.]', '', val)
        if val == '':
            return np.nan
        try:
            return float(val)
        except:
            return np.nan
    return val

for col in df.columns:
    if col not in text_cols:
        df[col] = df[col].apply(clean_currency_and_numbers)

### 4. Handle Missing/Null Values

In [ ]:
# Fill missing metrics with 0
for col in df.columns:
    if col not in text_cols:
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna('Unknown')

missing_pct = df.isnull().sum().max()
print(f"Max missing values in any column after cleaning: {missing_pct}%")

### 5. Save Cleaned Dataset

In [ ]:
output_path = 'military_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to {output_path}")